# PyIceberg 基礎分析ノートブック

`dlh_dev` カタログ配下の JEPX Silver テーブル（`base` / `block` / `area`）を PyIceberg 経由で読み込み、スキーマ確認・データ品質チェック・記述統計・可視化までの基礎的な分析を行う。

## 1. セットアップ

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import plotly.express as px
import polars as pl

WORKSPACE_ROOT = (
    Path.cwd().resolve().parents[1] if Path.cwd().name == "Jupyter" else Path("/workspace")
)
SRC_PATH = WORKSPACE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from common.iceberg import get_catalog

pl.Config.set_tbl_rows(20)

polars.config.Config

## 2. カタログへの接続

`configuration/iceberg/.pyiceberg.yaml` の `dlh_dev` カタログ（SQLite カタログ / RustFS ウェアハウス）に接続する。

In [2]:
catalog = get_catalog("dlh_dev")

for namespace in catalog.list_namespaces():
    for table in catalog.list_tables(namespace):
        print(".".join(table))

bronze.jepx_spot_price
bronze.occto_unit_generation_actuals
silver.jepx_spot_price_area
silver.jepx_spot_price_base
silver.jepx_spot_price_block
silver.occto_unit_generation_actuals


## 3. Silver テーブルの読み込み

JEPX スポット価格の Silver テーブル 3 種を Polars DataFrame として読み込む。

In [3]:
TABLE_IDENTIFIERS = {
    "base": "silver.jepx_spot_price_base",
    "block": "silver.jepx_spot_price_block",
    "area": "silver.jepx_spot_price_area",
}

frames: dict[str, pl.DataFrame] = {
    name: catalog.load_table(identifier).scan().to_polars()
    for name, identifier in TABLE_IDENTIFIERS.items()
}

df_base = frames["base"]
df_block = frames["block"]
df_area = frames["area"]

for name, df in frames.items():
    print(f"{name}: shape={df.shape}")

base: shape=(374400, 12)
block: shape=(374400, 12)
area: shape=(3364896, 10)


## 4. スキーマと期間の確認

In [4]:
for name, df in frames.items():
    print(f"--- {name} ---")
    print(df.schema)
    print()

--- base ---
Schema({'delivery_datetime': Datetime(time_unit='us', time_zone='UTC'), 'selling_bid_volume': Int64, 'purchase_bid_volume': Int64, 'contracted_volume': Int64, 'system_price': Decimal(precision=32, scale=3), 'delivery_date': Date, 'time_code': Int32, 'source_data': String, 'status': String, 'ingestion_time': Datetime(time_unit='us', time_zone='UTC'), 'ingestion_date': Date, 'execution_id': String})

--- block ---
Schema({'delivery_datetime': Datetime(time_unit='us', time_zone='UTC'), 'block_selling_bid_volume': Int64, 'block_selling_contracted_volume': Int64, 'block_purchase_bid_volume': Int64, 'block_purchase_contracted_volume': Int64, 'delivery_date': Date, 'time_code': Int32, 'source_data': String, 'status': String, 'ingestion_time': Datetime(time_unit='us', time_zone='UTC'), 'ingestion_date': Date, 'execution_id': String})

--- area ---
Schema({'delivery_datetime': Datetime(time_unit='us', time_zone='UTC'), 'area_name': String, 'area_price': Decimal(precision=32, scale=

In [5]:
for name, df in frames.items():
    date_range = df.select(
        pl.col("delivery_date").min().alias("min_date"),
        pl.col("delivery_date").max().alias("max_date"),
    )
    print(f"{name}: {date_range.row(0)}")

base: (datetime.date(2005, 4, 2), datetime.date(2026, 8, 9))
block: (datetime.date(2005, 4, 2), datetime.date(2026, 8, 9))
area: (datetime.date(2005, 4, 2), datetime.date(2026, 8, 9))


## 5. データ品質チェック

欠損値の件数と `status` 列の値の内訳を確認する。

In [6]:
for name, df in frames.items():
    print(f"--- {name}: null counts ---")
    print(df.null_count())
    print()

--- base: null counts ---
shape: (1, 12)
┌────────────┬────────────┬───────────┬───────────┬───┬────────┬───────────┬───────────┬───────────┐
│ delivery_d ┆ selling_bi ┆ purchase_ ┆ contracte ┆ … ┆ status ┆ ingestion ┆ ingestion ┆ execution │
│ atetime    ┆ d_volume   ┆ bid_volum ┆ d_volume  ┆   ┆ ---    ┆ _time     ┆ _date     ┆ _id       │
│ ---        ┆ ---        ┆ e         ┆ ---       ┆   ┆ u32    ┆ ---       ┆ ---       ┆ ---       │
│ u32        ┆ u32        ┆ ---       ┆ u32       ┆   ┆        ┆ u32       ┆ u32       ┆ u32       │
│            ┆            ┆ u32       ┆           ┆   ┆        ┆           ┆           ┆           │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪════════╪═══════════╪═══════════╪═══════════╡
│ 0          ┆ 0          ┆ 0         ┆ 0         ┆ … ┆ 0      ┆ 0         ┆ 0         ┆ 0         │
└────────────┴────────────┴───────────┴───────────┴───┴────────┴───────────┴───────────┴───────────┘

--- block: null counts ---
shape: (1, 12)
┌──────

In [7]:
for name, df in frames.items():
    print(f"--- {name}: status value counts ---")
    print(df["status"].value_counts())
    print()

--- base: status value counts ---
shape: (1, 2)
┌────────┬────────┐
│ status ┆ count  │
│ ---    ┆ ---    │
│ str    ┆ u32    │
╞════════╪════════╡
│ loaded ┆ 374400 │
└────────┴────────┘

--- block: status value counts ---
shape: (1, 2)
┌────────┬────────┐
│ status ┆ count  │
│ ---    ┆ ---    │
│ str    ┆ u32    │
╞════════╪════════╡
│ loaded ┆ 374400 │
└────────┴────────┘

--- area: status value counts ---
shape: (1, 2)
┌────────┬─────────┐
│ status ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ loaded ┆ 3364896 │
└────────┴─────────┘



## 6. システムプライスの記述統計

`base` テーブルの `system_price`（円/kWh）について基本統計量を確認する。

In [8]:
df_base.select(
    pl.col("system_price").mean().alias("mean"),
    pl.col("system_price").median().alias("median"),
    pl.col("system_price").std().alias("std"),
    pl.col("system_price").min().alias("min"),
    pl.col("system_price").max().alias("max"),
    pl.col("system_price").quantile(0.25).alias("p25"),
    pl.col("system_price").quantile(0.75).alias("p75"),
)

mean,median,std,min,max,p25,p75
f64,f64,f64,"decimal[32,3]","decimal[32,3]",f64,f64
11.39994,10.05,7.585096,0.010,251.000,7.4,13.9


## 7. 日次平均システムプライスの推移

`delivery_date` ごとに `system_price` を平均し、時系列で可視化する。

In [9]:
daily_price = (
    df_base.group_by("delivery_date")
    .agg(pl.col("system_price").mean().alias("avg_system_price"))
    .sort("delivery_date")
)

fig = px.line(
    daily_price,
    x="delivery_date",
    y="avg_system_price",
    title="日次平均システムプライスの推移",
    labels={"delivery_date": "受渡日", "avg_system_price": "平均システムプライス (円/kWh)"},
)
fig.show()

## 8. エリア別価格分析

`area` テーブルを使い、エリアごとの平均価格を比較する。

In [10]:
area_avg_price = (
    df_area.group_by("area_name")
    .agg(
        pl.col("area_price").mean().alias("avg_area_price"),
        pl.col("area_price").std().alias("std_area_price"),
    )
    .sort("avg_area_price", descending=True)
)

area_avg_price

area_name,avg_area_price,std_area_price
str,f64,f64
"""tokyo""",12.184524,8.441038
"""hokkaido""",12.089498,8.436691
"""tohoku""",11.865653,8.249839
"""chubu""",11.431711,7.688972
"""hokuriku""",11.182713,7.518037
"""kansai""",11.157682,7.509131
"""chugoku""",11.086043,7.456092
"""shikoku""",10.896849,7.476443
"""kyushu""",10.553953,7.2563


In [11]:
fig = px.bar(
    area_avg_price,
    x="area_name",
    y="avg_area_price",
    error_y="std_area_price",
    title="エリア別 平均価格",
    labels={"area_name": "エリア", "avg_area_price": "平均価格 (円/kWh)"},
)
fig.show()

## 9. 約定量（Volume）の推移

`base` テーブルの売り入札量・買い入札量・約定量を日次集計して比較する。

In [12]:
daily_volume = (
    df_base.group_by("delivery_date")
    .agg(
        pl.col("selling_bid_volume").sum().alias("selling_bid_volume"),
        pl.col("purchase_bid_volume").sum().alias("purchase_bid_volume"),
        pl.col("contracted_volume").sum().alias("contracted_volume"),
    )
    .sort("delivery_date")
)

fig = px.line(
    daily_volume,
    x="delivery_date",
    y=["selling_bid_volume", "purchase_bid_volume", "contracted_volume"],
    title="日次 売り/買い入札量・約定量の推移",
    labels={"delivery_date": "受渡日", "value": "数量 (kWh)", "variable": "指標"},
)
fig.show()

## 10. 約定量とシステムプライスの相関

`contracted_volume` と `system_price` の関係を散布図と相関係数で確認する。

In [13]:
correlation = df_base.select(
    pl.corr("contracted_volume", "system_price").alias("corr_contracted_volume_vs_price")
)
correlation

corr_contracted_volume_vs_price
f64
0.129207


In [14]:
fig = px.scatter(
    df_base.sample(n=min(5000, df_base.height), seed=0),
    x="contracted_volume",
    y="system_price",
    opacity=0.4,
    title="約定量 vs システムプライス",
    labels={"contracted_volume": "約定量 (kWh)", "system_price": "システムプライス (円/kWh)"},
)
fig.show()

## 11. 今後7日間（30分単位）の約定量予測

`contracted_volume` は 2005 年以降の長期データを含み、市場規模自体が年々大きく変化しているため、直近 90 日間の実績のみを学習対象とする。特徴量は 30 分スロットを表す `time_code` と曜日 `day_of_week` とし、`RandomForestRegressor` で日中・週内の季節パターンを学習し、最終実績日の翌日から 7 日間（48 スロット × 7 日 = 336 コマ）を予測する。

In [15]:
from datetime import datetime, timedelta

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

FORECAST_HORIZON_DAYS = 7
TRAIN_WINDOW_DAYS = 90
HOLDOUT_DAYS = 7
FEATURE_COLUMNS = ["time_code", "day_of_week"]
TARGET_COLUMN = "contracted_volume"

forecast_base = (
    df_base.select(["delivery_date", "time_code", "contracted_volume"])
    .drop_nulls()
    .with_columns(pl.col("delivery_date").dt.weekday().alias("day_of_week"))
    .sort(["delivery_date", "time_code"])
)

last_date = forecast_base["delivery_date"].max()
train_window_start = last_date - timedelta(days=TRAIN_WINDOW_DAYS)
recent = forecast_base.filter(pl.col("delivery_date") >= train_window_start)

recent.tail(5)

delivery_date,time_code,contracted_volume,day_of_week
date,i32,i64,i8
2026-08-09,44,21727550,7
2026-08-09,45,22195000,7
2026-08-09,46,21756350,7
2026-08-09,47,21356200,7
2026-08-09,48,20865600,7


### 11.1 ホールドアウト検証

直近 90 日のうち最後の 7 日間をホールドアウトとして、モデルの予測精度（MAE / MAPE）を確認する。

In [16]:
holdout_start = last_date - timedelta(days=HOLDOUT_DAYS - 1)
train_set = recent.filter(pl.col("delivery_date") < holdout_start)
holdout_set = recent.filter(pl.col("delivery_date") >= holdout_start)

validation_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
validation_model.fit(train_set[FEATURE_COLUMNS].to_pandas(), train_set[TARGET_COLUMN].to_pandas())

holdout_actual = holdout_set[TARGET_COLUMN].to_pandas()
holdout_pred = validation_model.predict(holdout_set[FEATURE_COLUMNS].to_pandas())

holdout_mae = mean_absolute_error(holdout_actual, holdout_pred)
holdout_mape = ((holdout_actual - holdout_pred).abs() / holdout_actual).mean() * 100

print(f"直近{HOLDOUT_DAYS}日ホールドアウト MAE  : {holdout_mae:,.0f} kWh")
print(f"直近{HOLDOUT_DAYS}日ホールドアウト MAPE : {holdout_mape:.2f}%")

直近7日ホールドアウト MAE  : 4,850,126 kWh
直近7日ホールドアウト MAPE : 17.94%


### 11.2 今後7日間の予測

直近 90 日全体でモデルを再学習し、最終実績日の翌日から 7 日間 × 48 スロット分の特徴量を組み立てて予測する。

In [17]:
final_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
final_model.fit(recent[FEATURE_COLUMNS].to_pandas(), recent[TARGET_COLUMN].to_pandas())

time_codes_sorted = sorted(forecast_base["time_code"].unique().to_list())
SLOTS_PER_DAY = len(time_codes_sorted)
MINUTES_PER_SLOT = (24 * 60) // SLOTS_PER_DAY

future_rows = []
for day_offset in range(1, FORECAST_HORIZON_DAYS + 1):
    future_date = last_date + timedelta(days=day_offset)
    for slot_index, time_code in enumerate(time_codes_sorted):
        future_rows.append(
            {
                "delivery_date": future_date,
                "time_code": time_code,
                "day_of_week": future_date.isoweekday(),
                "plot_datetime": datetime.combine(future_date, datetime.min.time())
                + timedelta(minutes=slot_index * MINUTES_PER_SLOT),
            }
        )

future_frame = pl.DataFrame(future_rows)
future_frame = future_frame.with_columns(
    pl.Series(
        "predicted_contracted_volume",
        final_model.predict(future_frame[FEATURE_COLUMNS].to_pandas()),
    )
)

future_frame.select("delivery_date", "time_code", "predicted_contracted_volume").head(10)

delivery_date,time_code,predicted_contracted_volume
date,i64,f64
2026-08-10,1,1.7540e7
2026-08-10,2,1.7126e7
2026-08-10,3,1.7086e7
2026-08-10,4,1.6991e7
2026-08-10,5,1.7161e7
2026-08-10,6,1.7213e7
2026-08-10,7,1.7335e7
2026-08-10,8,1.7442e7
2026-08-10,9,1.7701e7


### 11.3 実績と予測の可視化

直近 14 日間の実績と、今後 7 日間（30 分単位）の予測を並べて表示する。

In [18]:
history_window = recent.filter(pl.col("delivery_date") >= last_date - timedelta(days=14)).to_pandas()
time_code_to_slot_index = {time_code: idx for idx, time_code in enumerate(time_codes_sorted)}
history_window["plot_datetime"] = history_window.apply(
    lambda row: datetime.combine(row["delivery_date"], datetime.min.time())
    + timedelta(minutes=time_code_to_slot_index[row["time_code"]] * MINUTES_PER_SLOT),
    axis=1,
)

history_plot = history_window[["plot_datetime", "contracted_volume"]].rename(
    columns={"contracted_volume": "value"}
)
history_plot["series"] = "実績"

forecast_plot = (
    future_frame.select("plot_datetime", "predicted_contracted_volume")
    .to_pandas()
    .rename(columns={"predicted_contracted_volume": "value"})
)
forecast_plot["series"] = "予測"

combined_plot = pd.concat([history_plot, forecast_plot], ignore_index=True)

fig = px.line(
    combined_plot,
    x="plot_datetime",
    y="value",
    color="series",
    title="約定量: 直近14日間の実績 と 今後7日間(30分単位)の予測",
    labels={"plot_datetime": "日時", "value": "約定量 (kWh)", "series": ""},
)
fig.show()